# LongFlow P1 — caching + gate check (1K/5K/decode/listen)

Runtime: **L4 GPU**. Pre-registered criteria: `experiments/p1_flow_head/NOTES.md`.

Upload: drag **`longflow_bundle.zip`** (repo root on the Mac; regenerate with `git archive -o longflow_bundle.zip HEAD src configs`) into the Files panel. The cache lives on **Google Drive**, so a VM recycle costs nothing — re-run the cold start and the caching loop resumes where it left off.

In [ ]:
# ===== COLD START (idempotent) =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time
CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists("/content/longflow_bundle.zip"):
    print("DRAG longflow_bundle.zip INTO THE FILES PANEL, then re-run this cell")
else:
    !cd /content && unzip -q -o longflow_bundle.zip
    sys.path.insert(0, "/content")
    from src.cache.capture import SampleCapture, save_utterance
    print("READY")

## 1. Caching loop — resumable, safe to interrupt

Streams LibriTTS-R `train.clean.360`; per utterance: voice-prompt = its own first 3s, generate the transcript through unmodified VibeVoice with capture on, save the (hidden, latent) pairs to Drive. Skips anything already cached. Expect ~10–20s per utterance; run it as long as the session allows — **500+ cached is enough to gate**.

In [ ]:
import soundfile as sf
from pathlib import Path
from datasets import load_dataset

TARGET = 800
ds = load_dataset("mythicinfinity/libritts_r", "clean", split="train.clean.360", streaming=True)
done = {f.stem for f in Path(CACHE_DIR).glob("*.pt")}
print(f"resuming with {len(done)} already cached")
t0, skipped = time.time(), 0
for ex in ds:
    if len(done) >= TARGET:
        break
    uid = str(ex["id"]).replace("/", "_")
    if uid in done:
        continue
    text = ex["text_normalized"].strip()
    audio, sr = ex["audio"]["array"], ex["audio"]["sampling_rate"]
    if not (30 <= len(text) <= 180) or len(audio) < 3 * sr:
        skipped += 1
        continue
    sf.write("/content/_prompt.wav", audio[: 3 * sr], sr)
    inputs = processor(
        text=[f"Speaker 1: {text}\n"], voice_samples=[["/content/_prompt.wav"]],
        return_tensors="pt", padding=True,
    )
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    try:
        with SampleCapture(model) as cap, torch.inference_mode():
            model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
        utt = cap.to_utterance(uid, text, meta={"speaker": str(ex.get("speaker_id"))})
        save_utterance(utt, f"{CACHE_DIR}/{uid}.pt")
        done.add(uid)
    except Exception as e:
        print("skip", uid, repr(e)[:120])
    if len(done) % 25 == 0:
        rate = (time.time() - t0) / max(1, len(done))
        print(f"{len(done)}/{TARGET} cached  ({rate:.0f}s/utt, {skipped} filtered)")
print(f"DONE: {len(done)} utterances cached")

## 2. Roundtrip sanity — gate criterion (a), do NOT skip

Decode CACHED ground-truth latents through the frozen decoder. If this doesn't sound like normal VibeVoice speech, the capture or decode path is broken and training would be garbage-in.

In [ ]:
# How does generate() un-scale + decode latents? Print the source, then mirror it.
!grep -n "speech_scaling_factor\|acoustic_tokenizer.decode\|speech_bias" \
    /content/VibeVoice/vibevoice/modular/modeling_vibevoice_inference.py | head -12

In [ ]:
import glob
from IPython.display import Audio, display

def decode_latents(z):  # z: [T, d_latent] head-space, fp
    """Mirror generate()'s un-scaling then decode. ADJUST the un-scale line to
    match the grep output above if it differs."""
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi  # <- verify against source print
    for shape in (z.unsqueeze(0), z.unsqueeze(0).transpose(1, 2)):
        try:
            out = model.model.acoustic_tokenizer.decode(shape)
            wav = (out[0] if isinstance(out, tuple) else out)
            return wav.detach().float().cpu().numpy().squeeze()
        except Exception as e:
            print(f"decode attempt {tuple(shape.shape)} failed: {repr(e)[:150]}")
    raise RuntimeError("both decode shapes failed — paste the errors to Claude")

f = sorted(glob.glob(f"{CACHE_DIR}/*.pt"))[0]
d = torch.load(f, weights_only=True)
print(d["utt_id"], "|", d["text"][:80], "| frames:", d["latent"].shape[0])
wav = decode_latents(d["latent"].float())
sf.write("/content/roundtrip.wav", wav, 24000)
display(Audio("/content/roundtrip.wav"))
print("LISTEN: does this sound like normal VibeVoice speech saying the text above?")

## 3. Train the flow head — 5K steps (~10–15 min on L4)

In [ ]:
# copy cache local first (Drive IO is slow for many small reads)
!mkdir -p /content/cache_local && cp {CACHE_DIR}/*.pt /content/cache_local/
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import load_pairs, save_checkpoint, train

data = load_pairs("/content/cache_local")
print(f"pairs: {data.hidden.shape[0]}  d_model={data.d_model}  d_latent={data.d_latent}")
head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
print(f"head params: {head.param_count()/1e6:.2f}M")
out = train(head, data, steps=5000, batch_size=1024, lr=2e-4,
            ema_decay=0.999, device="cuda", log_every=500)
save_checkpoint(f"{CKPT_DIR}/gate_5k.pt", head, out["ema"], data, step=5000)
print("checkpoint saved to Drive")

## 4. Decode flow-head samples — gate criterion (c), LISTEN

For a few utterances: decode ground-truth latents (teacher) vs 4-NFE flow-head samples from the same conditions. A/B them.

In [ ]:
from src.flow_head.trainer import load_checkpoint, sample_latents
head_ema, lat_mean, lat_std = load_checkpoint(f"{CKPT_DIR}/gate_5k.pt")
head_ema = head_ema.to("cuda")
os.makedirs("/content/gate_audio", exist_ok=True)

files = sorted(glob.glob("/content/cache_local/*.pt"))
for f in files[:3] + files[-2:]:
    d = torch.load(f, weights_only=True)
    tag = d["utt_id"]
    for nfe in (4, 16):
        z = sample_latents(head_ema, d["hidden"].float(), lat_mean, lat_std, nfe=nfe)
        sf.write(f"/content/gate_audio/{tag}_flow{nfe}.wav", decode_latents(z), 24000)
    sf.write(f"/content/gate_audio/{tag}_teacher.wav", decode_latents(d["latent"].float()), 24000)
    print(tag, "|", d["text"][:70])
    for suffix in ("teacher", "flow4", "flow16"):
        print(" ", suffix)
        display(Audio(f"/content/gate_audio/{tag}_{suffix}.wav"))

In [ ]:
import zipfile
with zipfile.ZipFile("/content/gate_audio.zip", "w") as z:
    z.write("/content/roundtrip.wav", "roundtrip.wav")
    for f in os.listdir("/content/gate_audio"):
        z.write(f"/content/gate_audio/{f}", f)
from google.colab import files
files.download("/content/gate_audio.zip")